# Model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ComponentEmbedding(nn.Module):
    """Embeds circuit components into a latent space"""
    
    def __init__(self, num_component_types, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_component_types, embedding_dim)
        
    def forward(self, component_types):
        """
        Args:
            component_types: Tensor of component type indices [batch_size, num_nodes] or [num_nodes]
        Returns:
            Component embeddings [batch_size, num_nodes, embedding_dim] or [num_nodes, embedding_dim]
        """
        return self.embedding(component_types)


class GraphAttention(nn.Module):
    """Multi-head graph attention layer for processing the adjacency matrix with edge bias."""
    
    def __init__(self, input_dim, output_dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = output_dim // num_heads
        assert output_dim % num_heads == 0, "output_dim must be divisible by num_heads"
        
        # Query, Key, Value projections
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        
        # Output projection
        self.output_proj = nn.Linear(output_dim, output_dim)
        
        # Edge feature integration: projects a single scalar edge feature to each head's bias
        self.edge_proj = nn.Linear(1, self.num_heads)
        
    def forward(self, node_features, adjacency):
        """
        Args:
            node_features: Node feature tensor of shape [batch_size, num_nodes, input_dim]
            adjacency: Adjacency matrix tensor of shape [batch_size, num_nodes, num_nodes]
        Returns:
            Updated node features of shape [batch_size, num_nodes, output_dim]
        """
        batch_size = node_features.size(0)
        num_nodes = node_features.size(1)

        # Project inputs to queries, keys, and values and then reshape for multi-head attention
        # Each projection now yields shape: [batch_size, num_nodes, output_dim]
        q = self.query(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        k = self.key(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        v = self.value(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        
        # Rearrange for attention: [batch_size, num_heads, num_nodes, head_dim]
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 3, 1)  # for dot-product attention, key is transposed on last two dims
        v = v.permute(0, 2, 1, 3)
        
        # Compute raw attention scores with scaling
        attn_scores = torch.matmul(q, k) / (self.head_dim ** 0.5)  # [batch_size, num_heads, num_nodes, num_nodes]
        
        # Incorporate edge information:
        # Expecting adjacency shape [batch_size, num_nodes, num_nodes]
        # Unsqueeze last dimension and project edge feature per head
        edge_bias = self.edge_proj(adjacency.unsqueeze(-1))  # [batch_size, num_nodes, num_nodes, num_heads]
        edge_bias = edge_bias.permute(0, 3, 1, 2)  # [batch_size, num_heads, num_nodes, num_nodes]
        attn_scores = attn_scores + edge_bias
        
        # Mask out non-adjacent nodes:
        # mask = (adjacency == 0).unsqueeze(1)  # [batch_size, 1, num_nodes, num_nodes]
        # Mask last row and column for self-attention
        # mask[:, :, -1, :] = True
        # mask[:, :, :, -1] = True

        # attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        #attn_scores = attn_scores.masked_fill(mask, float(-1000000.0))
        
        # Compute normalized attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)
        #print(attn_weights)
        # Weighted sum over values
        out = torch.matmul(attn_weights, v)  # [batch_size, num_heads, num_nodes, head_dim]
        
        # Reshape back: first bring num_nodes back to dim 1
        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.view(batch_size, num_nodes, self.num_heads * self.head_dim)  # [batch_size, num_nodes, output_dim]
        
        # Final projection
        out = self.output_proj(out)
        
        return out


class CircuitGraphTransformer(nn.Module):
    """Circuit graph transformer model using graph attention layers"""
    
    def __init__(self, num_component_types, hidden_dim=256, num_layers=6, num_heads=8, dropout=0.1):
        super().__init__()
        
        # Component embedding layer
        self.component_embedding = ComponentEmbedding(num_component_types, hidden_dim)
        
        # Positional encoding: supports up to 330 nodes
        self.position_embedding = nn.Parameter(torch.zeros(1, 330, hidden_dim))
        
        # Build a sequence of transformer layers with attention and feed-forward components
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attention': GraphAttention(hidden_dim, hidden_dim, num_heads=num_heads),
                'norm1': nn.LayerNorm(hidden_dim),
                'ffn': nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 4),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 4, hidden_dim),
                    nn.Dropout(dropout)
                ),
                'norm2': nn.LayerNorm(hidden_dim)
            })
            for _ in range(num_layers)
        ])
        
        # Edge predictor network that uses concatenated node pairs to predict edge probabilities
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, component_types, adjacency_matrix):
        """
        Args:
            component_types: Component type indices, shape [num_nodes] or [batch_size, num_nodes]
            adjacency_matrix: Adjacency matrix, shape [num_nodes, num_nodes] or [batch_size, num_nodes, num_nodes]
        Returns:
            Node features tensor: [batch_size, num_nodes, hidden_dim]
        """
        # Add batch dimension if inputs are for a single graph.
        if component_types.dim() == 1:
            component_types = component_types.unsqueeze(0)  # [1, num_nodes]
        if adjacency_matrix.dim() == 2:
            adjacency_matrix = adjacency_matrix.unsqueeze(0)  # [1, num_nodes, num_nodes]
        
        # Obtain component embeddings and add positional embeddings.
        node_features = self.component_embedding(component_types)  # [batch_size, num_nodes, hidden_dim]
        num_nodes = node_features.size(1)
        node_features = node_features + self.position_embedding[:, :num_nodes, :]
        # Process through each transformer layer
        for layer in self.layers:
            attn_output = layer['attention'](node_features, adjacency_matrix)
            node_features = layer['norm1'](node_features + attn_output)
            
            ffn_output = layer['ffn'](node_features)
            node_features = layer['norm2'](node_features + ffn_output)
        
        return node_features
        
    def predict_new_edges(self, component_types, adjacency_matrix, new_component_type):
        """
        Predicts connection probabilities between a new component and existing components.
        
        Args:
            component_types: Component type indices, shape [num_nodes] or [batch_size, num_nodes]
            adjacency_matrix: Adjacency matrix, shape [num_nodes, num_nodes] or [batch_size, num_nodes, num_nodes]
            new_component_type: Type index of the new component to add
        Returns:
            Edge probabilities for the new component: Tensor of shape [num_nodes]
        """
        # Get node embeddings
        node_features = self.forward(component_types, adjacency_matrix)
        # If using a single graph, remove the batch dimension.
        if node_features.size(0) == 1:
            node_features = node_features[0]  # [num_nodes, hidden_dim]
        
        # Get embedding for the new component, shape [1, hidden_dim]
        new_component_embedding = self.component_embedding(
            torch.tensor([new_component_type], device=component_types.device)
        )
        
        edge_probs = []
        for i in range(node_features.size(0)):
            # Concatenate features of existing node and new component
            pair_features = torch.cat([node_features[i], new_component_embedding[0]], dim=-1)
            edge_prob = self.edge_predictor(pair_features)  # Output shape: [1]
            edge_probs.append(edge_prob)
        
        return torch.cat(edge_probs)
    def get_component_embeddings(self,component,model_device):
        component_embedding = self.component_embedding(
            torch.tensor([component], device=model_device)
        )
        return component_embedding
        
class edge_preds(nn.Module):
    """Edge prediction model for predicting connections between components."""
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

### Model handeling class

In [2]:

class CircuitGraphPredictor:
    """High-level interface for the circuit graph prediction model"""
    
    def __init__(self, model_path=None, num_component_types=830, hidden_dim=256):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        #self.device = torch.device('cpu')
        self.model = CircuitGraphTransformer(
            num_component_types=num_component_types,
            hidden_dim=hidden_dim
        ).to(self.device)
        
        if model_path:
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        
    def predict_new_connections(self, components, adjacency_matrix, new_component_type, threshold=0.5):
        """
        Predicts connections for a new component to be added to the circuit.
        
        Args:
            components: List of component type indices
            adjacency_matrix: Current adjacency matrix as a numpy array or list [n x n]
            new_component_type: Type index of the new component
            threshold: Threshold probability to decide on edge creation
        
        Returns:
            New connections for the new component: Numpy array of shape [num_nodes] with 0/1 values.
        """
        # Convert inputs to tensors
        component_types = torch.tensor(components, dtype=torch.long).to(self.device)
        adj_matrix = torch.tensor(adjacency_matrix, dtype=torch.float).to(self.device)
        
        # Predict edge probabilities using the model without gradient calculations.
        with torch.no_grad():
            edge_probs = self.model.predict_new_edges(
                component_types, adj_matrix, new_component_type
            )
        
        # Apply the threshold to obtain binary edge predictions
        new_connections = (edge_probs >= threshold).float().cpu().numpy()
        
        return new_connections
    def train1(self, dataloader, num_epochs=10, learning_rate=1e-4):
        import tqdm
        """
        Train the model on circuit graph data.
        
        Args:
            dataloader: DataLoader providing (component_types, adjacency_matrix, target_edges) tuples
            num_epochs: Number of training epochs
            learning_rate: Learning rate for optimization
        """
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        loss_fn = nn.BCELoss()
        self.model.train()  # Set the model to training mode
       
        for epoch in range(num_epochs):
            total_loss = 0
            h=0
            loss=0
            dataloader=tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} ", unit="batch")
            for batch in dataloader :

                component_types, adjacency_matrix,seq = batch
    
                print(adjacency_matrix.shape,component_types.shape)
                h+=1
                component_types = component_types.to(self.device)
                adjacency_matrix = adjacency_matrix.to(self.device)
                
                # Get node embeddings
                node_features = self.model(component_types, adjacency_matrix)
                # Calculate pairwise edge predictions
                edge_preds = []
                for i in range(len(component_types)):
                    for j in range(len(component_types)):
                        if i != j:  # Don't predict self-loops
                            pair_features = torch.cat([node_features[:,i,:], node_features[:,j,:]], dim=-1)
                            edge_prob = self.model.edge_predictor(pair_features)
                            edge_preds.append(edge_prob)
                
                edge_preds = torch.cat(edge_preds).to(self.device)

                target_edges=[]
                for i in range(len(component_types)):
                    for j in range(len(component_types)):
                        if i != j:  # Don't predict self-loops
                            target_edges.append(adjacency_matrix[i][j])
                target_edges = torch.tensor([target_edges], dtype=torch.float).view(1,1).to(self.device)
                # Convert target edges to the same device as edge_preds
                
                # Calculate loss
                loss = loss_fn(edge_preds, target_edges)
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                # print(f"{h} Batch Loss: {loss.item():.4f}")
            
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")

            # Example usage of the training function
            # Assuming `dataloader` is already defined and provides the required data

    def train1(self, dataloader, num_epochs=10, learning_rate=1e-4):
        import tqdm
        """
        Train the model on circuit graph data.
        
        Args:
            dataloader: DataLoader providing (component_types, adjacency_matrix, target_edges) tuples
            num_epochs: Number of training epochs
            learning_rate: Learning rate for optimization
        """
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        loss_fn = nn.BCELoss()
        self.model.train()  # Set the model to training mode
        
        for epoch in range(num_epochs):
            total_loss = 0
            dataloader = tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")
            
            for batch in dataloader:
                component_types, adjacency_matrix, _ = batch  # seq is unused here
                
                component_types = component_types.to(self.device)        # (B, N)
                adjacency_matrix = adjacency_matrix.to(self.device)      # (B, N, N)
                B, N = component_types.size()

                # Forward pass
                node_features = self.model(component_types, adjacency_matrix)  # (B, N, D)

                # Prepare pairwise edge features for prediction
                edge_preds = []
                target_edges = []

                for b in range(B):
                    nf = node_features[b]        # (N, D)
                    adj = adjacency_matrix[b]    # (N, N)
                    for i in range(N):
                        for j in range(N):
                            if i != j:
                                pair_features = torch.cat([nf[i], nf[j]], dim=-1)  # (2D,)
                                edge_prob = self.model.edge_predictor(pair_features)  # (1,)
                                edge_preds.append(edge_prob)

                                target_edges.append(adj[i][j].unsqueeze(0))  # (1,)

                edge_preds = torch.stack(edge_preds).to(self.device)       # (B*(N^2 - N), 1)
                target_edges = torch.stack(target_edges).to(self.device)   # (B*(N^2 - N), 1)

                # Compute loss
                loss = loss_fn(edge_preds, target_edges)

                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")
    def train(self, dataloader, num_epochs=10, learning_rate=1e-4):
        import tqdm
        """
        Train the model on circuit graph data using GPU-optimized batching.
        
        Args:
            dataloader: DataLoader providing (component_types, adjacency_matrix, target_edges) tuples
            num_epochs: Number of training epochs
            learning_rate: Learning rate for optimization
        """
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        loss_fn = nn.BCELoss()
        self.model.train()

        for epoch in range(num_epochs):
            total_loss = 0
            dataloader = tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

            for batch in dataloader:
                component_types, adjacency_matrix, _ = batch  # seq unused

                component_types = component_types.to(self.device)        # (B, N)
                adjacency_matrix = adjacency_matrix.to(self.device)      # (B, N, N)
                B, N = component_types.size()

                # Forward pass: node embeddings
                node_features = self.model(component_types, adjacency_matrix)  # (B, N, D)
                
                # Generate all i≠j pairs in advance
                i_idx, j_idx = torch.meshgrid(torch.arange(N), torch.arange(N), indexing="ij")
                mask = (i_idx != j_idx).to(self.device)  # (N, N)
                i_idx = i_idx.to(self.device)  # Move to the same device as node_features
                j_idx = j_idx.to(self.device)

                i_idx = i_idx[mask].flatten()  # (N*(N-1),)
                j_idx = j_idx[mask].flatten()

                # Ensure indices are on the same device as the tensor being indexed

                # Broadcast and index
                i_idx = i_idx.unsqueeze(0).expand(B, -1)  # (B, N*(N-1))
                j_idx = j_idx.unsqueeze(0).expand(B, -1)

                batch_idx = torch.arange(B).unsqueeze(1).expand(-1, i_idx.size(1)).to(self.device)

                node_i = node_features[batch_idx, i_idx]  # (B, N*(N-1), D)
                node_j = node_features[batch_idx, j_idx]  # (B, N*(N-1), D)
                pair_features = torch.cat([node_i, node_j], dim=-1)  # (B, N*(N-1), 2D)
                # Flatten for predictor
                pair_features_flat = pair_features.view(-1, pair_features.size(-1))  # (B*(N*(N-1)), 2D)
                edge_preds = self.model.edge_predictor(pair_features_flat)  # (B*(N*(N-1)), 1)

                # Get targets
                target_edges = adjacency_matrix[batch_idx, i_idx, j_idx]  # (B, N*(N-1))
                target_edges = target_edges.view(-1, 1).float()  # (B*(N*(N-1)), 1)

                # Loss + Backprop
                loss = loss_fn(edge_preds, target_edges)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
            
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")

            


In [3]:
# predictor = CircuitGraphPredictor(num_component_types=892, hidden_dim=256)
# predictor.train(loader)

#### Example usage

In [4]:

# Example usage:
def create_dataset_from_circuits(circuit_graphs, component_libraries):
    """
    Create a dataset from a collection of circuit graphs for training the model.
    
    Args:
        circuit_graphs: List of (component_list, adjacency_matrix) tuples.
        component_libraries: Dictionary mapping component names to type indices.
        
    Returns:
        Dataset for training the model.
    """
    # Implementation depends on specific data format.
    pass


def main():
    # Initialize predictor with 50 component types.
    predictor = CircuitGraphPredictor(num_component_types=50)
    
    # Example components and corresponding adjacency matrix for a single graph.
    components = [0, 1, 2, 3]  # Component type indices
    adjacency_matrix = [
        [0, 1, 0, 1],  # Component 0 connects to 1 and 3
        [1, 0, 1, 0],  # Component 1 connects to 0 and 2
        [0, 1, 0, 1],  # Component 2 connects to 1 and 3
        [1, 0, 1, 0]   # Component 3 connects to 0 and 2
    ]
    
    # Predict new connections for a new component of type 4.
    new_component_type = 4
    new_connections = predictor.predict_new_connections(
        components, adjacency_matrix, new_component_type
    )
    
    print(f"Predicted connections for new component: {new_connections}")
    
    # The new adjacency matrix would include the new row and column corresponding to new_connections.
    # For example:
    # [
    #   [0, 1, 0, 1, new_connections[0]],
    #   [1, 0, 1, 0, new_connections[1]],
    #   [0, 1, 0, 1, new_connections[2]],
    #   [1, 0, 1, 0, new_connections[3]],
    #   [new_connections[0], new_connections[1], new_connections[2], new_connections[3], 0]
    # ]


#### Data loading

In [5]:
def collate_fn(batch):
    seqs, mats = zip(*batch)

    # Convert sequences to torch tensors
    seqs = [torch.tensor(seq, dtype=torch.long) for seq in seqs]

    # Convert adjacency matrices (NumPy -> PyTorch)
    mats = [torch.tensor(mat, dtype=torch.float32) for mat in mats]

    # Get max sizes
    max_seq_len = max(len(seq) for seq in seqs)
    max_nodes = max(mat.size(0) for mat in mats)

    # Pad sequences
    padded_seqs = torch.stack([
        F.pad(seq, (0, max_seq_len - len(seq)), value=0)
        for seq in seqs
    ])

    # Pad adjacency matrices
    padded_mats = torch.stack([
        F.pad(mat, (0, max_nodes - mat.size(1), 0, max_nodes - mat.size(0)), value=0)
        for mat in mats
    ])

    seq_lengths = torch.tensor([len(seq) for seq in seqs])

    return padded_seqs, padded_mats, seq_lengths



In [6]:
from sklearn.model_selection import train_test_split
from Circuits import Circuits
circuits=Circuits()

dataset = list(zip(circuits.component_indices, circuits.graphs))
# Split dataset into training and testing sets
train_data, test_data = train_test_split(dataset, test_size=0.1, random_state=42)
loader = torch.utils.data.DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collate_fn)


Loading dataset files...


# Current Training

In [7]:
predictor = CircuitGraphPredictor(num_component_types=892, hidden_dim=256)

In [8]:
# predictor=CircuitGraphPredictor(num_component_types=892,model_path="circuit_graph_predictor2.pth", hidden_dim=256)

In [9]:
predictor.train(loader)

Epoch 1/10:  48%|████▊     | 365/754 [00:46<00:49,  7.86batch/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Save the model's state dictionary
model_save_path = "circuit_graph_predictor2.pth"
torch.save(predictor.model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to circuit_graph_predictor2.pth


# Graph Prediction

In [ ]:
def generate_edges(edge_preds, component_types):
    """ 
    Generate edges based on edge predictions and component types.
    Args:
        edge_preds: Edge prediction tensor of shape [num_nodes, num_nodes]
        component_types: Component type indices of shape [num_nodes]
    Returns:
        adjacency_matrix: Generated adjacency matrix of shape [num_nodes, num_nodes]
    """
    a=torch.tensor(edge_preds)
    a=a.squeeze(-1)
    softmax = torch.nn.functional.softmax(a, dim=-1)
    # print(softmax)
    # Mark 1 for the top two values of each row
    top_two_indices = torch.topk(softmax, 2, dim=-1).indices
    marked = torch.zeros_like(softmax)
    marked.scatter_(-1, top_two_indices, 1)

    # Add a self-loop as zero to make the adjacency matrix n*n
    adjacency_matrix=[]
    for i in range(len(component_types)):
        adjacency_matrix.append(torch.cat((marked[i,:i],torch.tensor([0.0]),marked[i,i:])))
    adjacency_matrix = torch.stack(adjacency_matrix)
    return adjacency_matrix

def generate_edges_with_thrishold(edge_preds, component_types):
    """ 
    Generate edges based on edge predictions and component types.
    Args:
        edge_preds: Edge prediction tensor of shape [num_nodes, num_nodes]
        component_types: Component type indices of shape [num_nodes]
    Returns:
        adjacency_matrix: Generated adjacency matrix of shape [num_nodes, num_nodes]
    """
    a=torch.tensor(edge_preds)
    a=a.squeeze(-1)
    # softmax = torch.nn.functional.softmax(a, dim=-1)
    # print(softmax)
    # Mark 1 for the top two values of each row
    threshold = 0.1
    mask = a >= threshold
    out = torch.zeros_like(a)
    out[mask] = 1
    # top_two_indices = torch.topk(softmax, 2, dim=-1).indices
    # marked = torch.zeros_like(softmax)
    # marked.scatter_(-1, top_two_indices, 1)

    # Add a self-loop as zero to make the adjacency matrix n*n
    adjacency_matrix=[]
    for i in range(len(component_types)):
        adjacency_matrix.append(torch.cat((mask[i,:i],torch.tensor([0.0]),mask[i,i:])))
    adjacency_matrix = torch.stack(adjacency_matrix)
    return adjacency_matrix

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
component_types =torch.randint(0, 892, (10,), device=device)  # Random component types for testing
adjacency_matrix = torch.zeros((len(component_types),len(component_types))).to(device)
# Create a random adjacency matrix for testing



In [ ]:
def predict_edges(model, component_types, adjacency_matrix,device="cpu"):
    component_types = component_types.to(device)
    adjacency_matrix = adjacency_matrix.to(device)
    
    # Get node embeddings
    node_features = model(component_types, adjacency_matrix)
    # Calculate pairwise edge predictions
    edge_preds = []
    for i in range(len(component_types)):
        row_preds=[]
        for j in range(len(component_types)):
            if i != j:  # Don't predict self-loops
                pair_features = torch.cat([node_features[:,i,:], node_features[:,j,:]], dim=-1)
                edge_prob = model.edge_predictor(pair_features)
                row_preds.append(edge_prob)
        edge_preds.append(row_preds)
    return edge_preds

In [ ]:
graphdataset,textdataset = Circuits().data_lodder_withoutpading()

Loading dataset files...


In [ ]:
a=graphdataset[12],textdataset[12]
component_types =torch.tensor(a[1]).to(device)
adjacency_matrix = torch.zeros((len(component_types),len(component_types))).to(device)

C:\Users\MSI\AppData\Local\Temp\ipykernel_11192\1671230171.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  component_types =torch.tensor(a[1]).to(device)


In [ ]:
from tqdm import tqdm
predictor=CircuitGraphPredictor(num_component_types=892,model_path="circuit_graph_predictor.pth", hidden_dim=256)

model=predictor.model.to(device)
tqdm_iter = tqdm(range(1000), desc="Generating Adjacency Matrix")
for _ in tqdm_iter:
    edge_preds = predict_edges(model, component_types, adjacency_matrix,device=device)
    adjacency_matrix = generate_edges(edge_preds, component_types).to(device)
print("Generated Adjacency Matrix:")
print(adjacency_matrix)


NameError: name 'CircuitGraphPredictor' is not defined

In [ ]:
adjacency_matrix = adjacency_matrix.cpu().numpy()
adjacency_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

## Preformance

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compare_adjacency_matrices(predicted: np.ndarray, ground_truth: np.ndarray):
    """
    Compare two adjacency matrices using accuracy, precision, recall, and F1 score.
    
    Args:
        predicted (np.ndarray): The predicted adjacency matrix.
        ground_truth (np.ndarray): The ground truth adjacency matrix.

    Returns:
        dict: Dictionary containing accuracy, precision, recall, and F1 score.
    """
    def custom_precision_score(y_true, y_pred, fp_weight=2.0):
        from sklearn.metrics import confusion_matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        weighted_precision = tp*2 / (tp*2 +  fp + 1e-8)
        return weighted_precision

    assert predicted.shape == ground_truth.shape, "Matrices must be the same shape."

    # Flatten the matrices to compare element-wise
    y_pred = predicted.flatten()
    y_true = ground_truth.flatten()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": custom_precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0)
    }

# Example usage
if __name__ == "__main__":
    # Example adjacency matrices (binary, symmetric if undirected)
    ground_truth =torch.tensor(a[0])

    predicted = torch.tensor(adjacency_matrix).to("cpu")
    results = compare_adjacency_matrices(predicted, ground_truth)
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")


accuracy: 0.9321
precision: 0.1026
recall: 0.0333
f1_score: 0.0412


C:\Users\MSI\AppData\Local\Temp\ipykernel_11192\3631384064.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ground_truth =torch.tensor(a[0])
